# Notebook 03: DQI Branch-Mixture Microscope (Exploratory, Non-Authoritative)

**Reference:** Jordan et al., "Optimization by Decoded Quantum Interferometry," [arXiv:2408.08292](https://arxiv.org/abs/2408.08292) (2024)

> **Exploratory scope note**: this notebook is a microscope-style diagnostic for branch-mixture DQI behavior, not a canonical reporting layer.\n> It uses the branch-mixture statevector simplification with fixed-weight branches weighted by `|alpha_t|^2`, and includes a tiny-instance coherent reference check under the repo's projective/postselected surrogate model.\n> Coherent results here are **not** a claim of fully reversible physical decoder simulation, and outputs are not authoritative stage-report figures.


In [1]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

try:
    from notebooks._helpers import (
        repo_root,
        load_instance,
        verification_card,
        format_benchmark_table,
        unavailable_panel_table,
    )
except ModuleNotFoundError:
    from _helpers import (
        repo_root,
        load_instance,
        verification_card,
        format_benchmark_table,
        unavailable_panel_table,
    )

ROOT = repo_root()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.reduction import build_surrogate, surrogate_objective, surrogate_faithfulness
from src.weights import uniform_weights, optimal_weights, max_safe_ell
from src.dqi_state import (
    prepare_dicke_state,
    apply_phase_kick,
    compute_syndrome_state,
    decode_and_uncompute,
    hadamard_transform_state,
    sample_from_probabilities,
)
from src.decoder_bruteforce import BoundedDistanceDecoder

BENCHMARK_SEED = 42
BENCHMARK_N_SAMPLES = 1000
np.random.seed(BENCHMARK_SEED)

def canonical_benchmark_k(prob):
    if prob.n_features == 5:
        return 15
    return min(12, (2 ** prob.n_bits) - 1)


 **Five-stage DQI path used here.**
 
 1) Prepare Dicke superposition over low-weight error patterns.
 2) Apply phase kick according to target parity vector $v$.
 3) Compute syndrome $B^T y$.
 4) Decode and uncompute with bounded-distance decoder.
 5) Apply Hadamard on output register and measure candidates.
 
 **Benchmark alignment in this notebook:** `seed=42`, `n_samples=1000`, and per-instance `k`/`ell` match the canonical benchmark configuration.

In [2]:
def run_explicit_five_step(B, v, alpha, ell, n_samples=2000, seed=0):
    m, n = B.shape
    decoder = BoundedDistanceDecoder(B, ell)

    alpha_sq = np.abs(alpha[:ell+1]) ** 2
    alpha_total = float(np.sum(alpha_sq))
    probabilities = np.zeros(2 ** n, dtype=np.float64)
    success_prob = 0.0

    for t in range(ell + 1):
        branch = float(alpha_sq[t] / alpha_total) if alpha_total > 0 else 0.0
        if branch < 1e-15:
            continue

        state = prepare_dicke_state(m, t)
        state = apply_phase_kick(state, v)
        full_state = compute_syndrome_state(state, B)
        syndrome_state, success_t = decode_and_uncompute(full_state, B, decoder)
        final_state = hadamard_transform_state(syndrome_state)

        probabilities += branch * (np.abs(final_state) ** 2)
        success_prob += branch * success_t

    total = float(np.sum(probabilities))
    if total > 1e-15:
        probabilities /= total

    samples = sample_from_probabilities(probabilities, n_samples=n_samples, seed=seed)
    return samples, probabilities, float(success_prob)

**Note:** Full-space $\rho(F,G)$ is computed exactly here because these are tiny 6/8/10-bit instances.

In [3]:
instance_specs = [
    ('6-bit', 'data/instances/pricing_3feat_6bit.json'),
    ('8-bit', 'data/instances/pricing_4feat_8bit.json'),
    ('10-bit', 'data/instances/pricing_5feat_10bit.json'),
]

cards = []
missing_instances = []
for name, rel_path in instance_specs:
    instance_path = ROOT / rel_path
    if not instance_path.exists():
        missing_instances.append({"instance": name, "path": rel_path, "status": "missing"})
        continue

    prob = load_instance(instance_path)
    n_bits = prob.n_bits
    k = canonical_benchmark_k(prob)
    surrogate = build_surrogate(prob, k=k)
    B, v, w = surrogate['B'], surrogate['v'], surrogate['weights']
    m = B.shape[0]
    ell = min(3, max_safe_ell(m, n_bits))

    x_opt, f_opt = prob.brute_force_solve()
    rho, _ = surrogate_faithfulness(prob, k=k)

    display(Markdown(f'### Instance {name}: n={n_bits}, m={m}, k={k}, ell={ell}'))

    alpha_uniform = uniform_weights(ell)
    alpha_eig = optimal_weights(B, v, ell)

    run_stats = {}
    for alpha_label, alpha in [('uniform_alpha', alpha_uniform), ('current_eigenvector_alpha', alpha_eig)]:
        samples, probs, success_prob = run_explicit_five_step(
            B, v, alpha, ell, n_samples=BENCHMARK_N_SAMPLES, seed=BENCHMARK_SEED
        )
        F_vals = np.array([prob.evaluate(int(x)) for x in samples], dtype=np.float64)
        G_vals = np.array([surrogate_objective(int(x), B, v, w, n_bits) for x in samples], dtype=np.float64)

        best_idx_F = int(np.argmax(F_vals))
        best_idx_G = int(np.argmax(G_vals))
        best_sampled_F = float(F_vals[best_idx_F])
        best_sampled_G = float(G_vals[best_idx_G])
        F_top_surrogate = float(F_vals[best_idx_G])
        sampled_gap = float(best_sampled_F - F_top_surrogate)

        approx_ratio = best_sampled_F / f_opt if f_opt > 0 else 0.0

        run_stats[alpha_label] = {
            'success_prob': float(success_prob),
            'best_sampled_F': best_sampled_F,
            'best_sampled_G': best_sampled_G,
            'F_top_surrogate_ranked_sample': F_top_surrogate,
            'sampled_misranking_gap': sampled_gap,
            'approx_ratio': float(approx_ratio),
            'misranking_flag': bool(sampled_gap > 1e-9),
        }

    anchor = pd.DataFrame([{
        'true optimum F*': f_opt,
        'best sampled F (uniform)': run_stats['uniform_alpha']['best_sampled_F'],
        'best sampled F (eigenvector)': run_stats['current_eigenvector_alpha']['best_sampled_F'],
        'approx ratio uniform': run_stats['uniform_alpha']['approx_ratio'],
        'approx ratio eigenvector': run_stats['current_eigenvector_alpha']['approx_ratio'],
    }])
    display(anchor)

    card_metrics = {
        'instance': name,
        'n': n_bits,
        'm': m,
        'k': k,
        'ell': ell,
        'F_star': float(f_opt),
        'rho_FG_exact': float(rho),
        'uniform_success_prob': run_stats['uniform_alpha']['success_prob'],
        'uniform_best_F': run_stats['uniform_alpha']['best_sampled_F'],
        'uniform_best_G': run_stats['uniform_alpha']['best_sampled_G'],
        'uniform_top_rank_regret': run_stats['uniform_alpha']['sampled_misranking_gap'],
        'uniform_misranking_flag': run_stats['uniform_alpha']['misranking_flag'],
        'eigen_success_prob': run_stats['current_eigenvector_alpha']['success_prob'],
        'eigen_best_F': run_stats['current_eigenvector_alpha']['best_sampled_F'],
        'eigen_best_G': run_stats['current_eigenvector_alpha']['best_sampled_G'],
        'eigen_top_rank_regret': run_stats['current_eigenvector_alpha']['sampled_misranking_gap'],
        'eigen_misranking_flag': run_stats['current_eigenvector_alpha']['misranking_flag'],
    }

    card = verification_card(card_metrics)
    cards.append(card)
    display(card.T.rename(columns={0: 'value'}))

if missing_instances:
    display(unavailable_panel_table(
        panel='Instance inputs',
        reason='Some tiny pricing instance files are missing; exploratory coverage is partial.',
        details=', '.join(f"{r['instance']}:{r['path']}" for r in missing_instances),
    ))

if cards:
    all_cards = pd.concat(cards, ignore_index=True)
    display(Markdown('## Verification Card Summary (all loaded instances)'))
    all_cards
else:
    all_cards = pd.DataFrame()
    display(unavailable_panel_table(
        panel='Verification card summary',
        reason='No instance cards were generated because required input files are missing.',
    ))


### Instance 6-bit: n=6, m=12, k=12, ell=1

,true optimum F*,best sampled F (uniform),best sampled F (eigenvector),approx ratio uniform,approx ratio eigenvector
0,3000.0,3000.0,3000.0,1.0,1.0


,value
instance,6-bit
n,6
m,12
k,12
ell,1
F_star,3000.0
rho_FG_exact,0.912265
uniform_success_prob,1.0
uniform_best_F,3000.0
uniform_best_G,25954.6875


### Instance 8-bit: n=8, m=12, k=12, ell=2

,true optimum F*,best sampled F (uniform),best sampled F (eigenvector),approx ratio uniform,approx ratio eigenvector
0,2800.0,2800.0,2800.0,1.0,1.0


,value
instance,8-bit
n,8
m,12
k,12
ell,2
F_star,2800.0
rho_FG_exact,0.944901
uniform_success_prob,0.606061
uniform_best_F,2800.0
uniform_best_G,40080.546875


### Instance 10-bit: n=10, m=15, k=15, ell=3

,true optimum F*,best sampled F (uniform),best sampled F (eigenvector),approx ratio uniform,approx ratio eigenvector
0,3000.0,2500.0,2500.0,0.833333,0.833333


,value
instance,10-bit
n,10
m,15
k,15
ell,3
F_star,3000.0
rho_FG_exact,0.946191
uniform_success_prob,0.148352
uniform_best_F,2500.0
uniform_best_G,50128.808594


## Verification Card Summary (all loaded instances)

**Benchmark JSON comparison (sanity sync)**

The table below compares notebook-computed metrics against `results/benchmark_all.json` using the same configuration.

In [4]:
benchmark_path = ROOT / 'results' / 'benchmark_all.json'

if all_cards.empty:
    cmp_df = pd.DataFrame()
    display(unavailable_panel_table(
        panel='Benchmark JSON comparison',
        reason='No notebook instance cards available for benchmark comparison.',
    ))
elif not benchmark_path.exists():
    cmp_df = pd.DataFrame()
    display(unavailable_panel_table(
        panel='Benchmark JSON comparison',
        reason=f"Missing benchmark file: {benchmark_path}",
        details="Run 'python -m src.benchmark' to generate benchmark_all.json",
    ))
else:
    try:
        benchmark_records = json.loads(benchmark_path.read_text())
        benchmark_table = format_benchmark_table(benchmark_records).sort_values('n_bits')

        notebook_table = pd.DataFrame({
            'n_bits': all_cards['n'].astype(int),
            'notebook_uniform_ratio': [float(f / f_star) if f_star > 0 else 0.0 for f, f_star in zip(all_cards['uniform_best_F'], all_cards['F_star'])],
            'notebook_heuristic_ratio': [float(f / f_star) if f_star > 0 else 0.0 for f, f_star in zip(all_cards['eigen_best_F'], all_cards['F_star'])],
            'notebook_uniform_success': all_cards['uniform_success_prob'].astype(float),
            'notebook_heuristic_success': all_cards['eigen_success_prob'].astype(float),
        })

        cmp_df = notebook_table.merge(
            benchmark_table[[
                'n_bits',
                'dqi_uniform_ratio',
                'dqi_heuristic_ratio',
                'dqi_uniform_success_prob',
                'dqi_heuristic_success_prob',
            ]],
            on='n_bits',
            how='left',
        ).rename(columns={
            'dqi_uniform_ratio': 'benchmark_uniform_ratio',
            'dqi_heuristic_ratio': 'benchmark_heuristic_ratio',
            'dqi_uniform_success_prob': 'benchmark_uniform_success',
            'dqi_heuristic_success_prob': 'benchmark_heuristic_success',
        }).sort_values('n_bits')

        display(cmp_df)

        display(Markdown('### 10-bit one-row comparison'))
        display(cmp_df[cmp_df['n_bits'] == 10])
        
        # Note about expected discrepancies
        display(Markdown("""
**Note on discrepancies:** This notebook uses `run_explicit_five_step()` with fixed seed=42 and n_samples=1000,
while `benchmark_all.json` uses the full `DQIPipeline.run_with_metrics()`. Minor differences in:
- Success probability: Due to different branch-weighting calculations
- Best sampled F: Due to sampling variance even with same seed

These are **expected exploratory-vs-benchmark differences**, not errors. This notebook is a microscope diagnostic,
not the authoritative benchmark source.
"""))
    except Exception as exc:
        cmp_df = pd.DataFrame()
        display(unavailable_panel_table(
            panel='Benchmark JSON comparison',
            reason='Benchmark file exists but could not be parsed/normalized.',
            details=str(exc),
        ))

,n_bits,notebook_uniform_ratio,notebook_heuristic_ratio,notebook_uniform_success,notebook_heuristic_success,benchmark_uniform_ratio,benchmark_heuristic_ratio,benchmark_uniform_success,benchmark_heuristic_success
0,6,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,8,1.000000,1.000000,0.606061,0.522752,1.000000,1.000000,0.606061,0.522752
2,10,0.833333,0.833333,0.148352,0.142717,0.833333,0.833333,0.148352,0.142717


### 10-bit one-row comparison

,n_bits,notebook_uniform_ratio,notebook_heuristic_ratio,notebook_uniform_success,notebook_heuristic_success,benchmark_uniform_ratio,benchmark_heuristic_ratio,benchmark_uniform_success,benchmark_heuristic_success
2,10,0.833333,0.833333,0.148352,0.142717,0.833333,0.833333,0.148352,0.142717



**Note on discrepancies:** This notebook uses `run_explicit_five_step()` with fixed seed=42 and n_samples=1000,
while `benchmark_all.json` uses the full `DQIPipeline.run_with_metrics()`. Minor differences in:
- Success probability: Due to different branch-weighting calculations
- Best sampled F: Due to sampling variance even with same seed

These are **expected exploratory-vs-benchmark differences**, not errors. This notebook is a microscope diagnostic,
not the authoritative benchmark source.


**Interpretation:** These tiny instances allow exact verification of the surrogate/true-objective relationship. The cards expose both successful recovery behavior and misranking signals where top surrogate-ranked samples do not maximize the true objective.

## Alpha ablation and coherent-vs-mixture check (tiny-instance exact surrogate model)

In [5]:
from src.benchmark import run_benchmark

tiny_instance_path = ROOT / 'data/instances/pricing_3feat_6bit.json'
if not tiny_instance_path.exists():
    display(unavailable_panel_table(
        panel='Tiny-instance alpha/coherent check',
        reason=f"Missing pricing instance file: {tiny_instance_path.relative_to(ROOT)}",
    ))
else:
    tiny_prob = load_instance(tiny_instance_path)
    try:
        tiny_results = run_benchmark(
            tiny_prob,
            k=8,
            ell=2,
            n_dqi_samples=600,
            top_k_by_surrogate=10,
            seed=BENCHMARK_SEED,
        )

        rows = []
        for key in ['dqi_uniform_mixture', 'dqi_paper_mixture', 'dqi_heuristic_mixture']:
            r = tiny_results[key]
            rows.append({
                'run_key': key,
                'alpha_mode': r.get('alpha_mode'),
                'execution_mode': r.get('execution_mode'),
                'best_F': r.get('best_F'),
                'approx_ratio': r.get('approximation_ratio'),
                'success_prob': r.get('success_prob'),
                'coherent_supported': r.get('coherent_supported', False),
                'coherent_exact_small_instance': r.get('coherent_exact_small_instance', False),
                # Explicitly mark mixture rows as completed (not applicable for coherent-specific fields)
                'status': 'completed',
                'reason': 'mixture_mode',
                'joint_state_size': None,
                'cap_limit': None,
            })

        coh = tiny_results.get('dqi_paper_coherent', {})
        if coh.get('status') == 'unsupported':
            rows.append({
                'run_key': 'dqi_paper_coherent',
                'alpha_mode': coh.get('alpha_mode', 'paper'),
                'execution_mode': coh.get('execution_mode', 'coherent'),
                'best_F': None,
                'approx_ratio': None,
                'success_prob': None,
                'coherent_supported': coh.get('coherent_supported', False),
                'coherent_exact_small_instance': coh.get('coherent_exact_small_instance', False),
                'status': coh.get('status'),
                'reason': coh.get('reason'),
                'joint_state_size': coh.get('joint_state_size'),
                'cap_limit': coh.get('cap_limit'),
            })
        else:
            rows.append({
                'run_key': 'dqi_paper_coherent',
                'alpha_mode': coh.get('alpha_mode'),
                'execution_mode': coh.get('execution_mode'),
                'best_F': coh.get('best_F'),
                'approx_ratio': coh.get('approximation_ratio'),
                'success_prob': coh.get('success_prob'),
                'coherent_supported': coh.get('coherent_supported', False),
                'coherent_exact_small_instance': coh.get('coherent_exact_small_instance', False),
                'status': 'completed' if coh.get('best_F') is not None else 'unknown',
                'reason': 'coherent_executed' if coh.get('best_F') is not None else 'no_result',
                'joint_state_size': coh.get('joint_state_size'),
                'cap_limit': coh.get('cap_limit'),
            })

        alpha_exec_cmp = pd.DataFrame(rows)
        # Replace NaN with dash for unsupported/missing fields
        display_cols_to_dash = ["best_F", "approx_ratio", "success_prob", "joint_state_size", "cap_limit"]
        alpha_display = alpha_exec_cmp.copy()
        for col in display_cols_to_dash:
            if col in alpha_display.columns:
                alpha_display[col] = alpha_display[col].fillna("—")
        display(alpha_display)
        
        # Show coherent feasibility note
        coherent_row = alpha_exec_cmp[alpha_exec_cmp['execution_mode'] == 'coherent'].iloc[0] if not alpha_exec_cmp[alpha_exec_cmp['execution_mode'] == 'coherent'].empty else None
        if coherent_row is not None and coherent_row['status'] == 'unsupported':
            display(Markdown(f"""
**Coherent execution note:** The 6-bit instance with k=8, ell=2 produces a joint Hilbert space of 
{int(coherent_row['joint_state_size']):,} states, which exceeds the cap limit of {int(coherent_row['cap_limit']):,}.
This is expected for the projective/postselected surrogate model used in this notebook.
Smaller ell values or reduced k would be needed for coherent execution within memory constraints.
"""))

        pair_diag = pd.DataFrame([tiny_results.get('dqi_pairwise_diagnostics', {})])
        display(pair_diag.T.rename(columns={0: 'value'}))
    except Exception as exc:
        display(unavailable_panel_table(
            panel='Tiny-instance alpha/coherent check',
            reason='Tiny-instance benchmark execution failed in exploratory notebook.',
            details=str(exc),
        ))

,run_key,alpha_mode,execution_mode,best_F,approx_ratio,success_prob,coherent_supported,coherent_exact_small_instance,status,reason,joint_state_size,cap_limit
0,dqi_uniform_mixture,uniform,mixture,3000.0,1.0,0.678571,False,False,completed,mixture_mode,—,—
1,dqi_paper_mixture,paper,mixture,3000.0,1.0,0.571429,False,False,completed,mixture_mode,—,—
2,dqi_heuristic_mixture,heuristic,mixture,3000.0,1.0,0.613977,False,False,completed,mixture_mode,—,—
3,dqi_paper_coherent,paper,coherent,—,—,—,False,False,unsupported,coherent_hilbert_cap_exceeded,65536.0,4096.0



**Coherent execution note:** The 6-bit instance with k=8, ell=2 produces a joint Hilbert space of 
65,536 states, which exceeds the cap limit of 4,096.
This is expected for the projective/postselected surrogate model used in this notebook.
Smaller ell values or reduced k would be needed for coherent execution within memory constraints.


,value
delta_best_F_paper_vs_heuristic,0.0
delta_success_prob_paper_vs_heuristic,-0.042549
delta_best_F_paper_mixture_vs_coherent,None
tvd_paper_mixture_vs_coherent,None
same_top_sample_flag_paper_mixture_vs_coherent,None
coherent_comparison_attempted,False
